# Week 7 - 02: Building the RAG Pipeline
Week 6:
**Question -> Embedding -> ChromaDB -> Relevant chunks**

Week 7:
**Relevant chunks -> LLM -> Answer**

Complete RAG:
**Retrieve -> Augment -> Generate**

In [1]:
!pip install chromadb sentence-transformers anthropic

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Gemini API key


In [2]:
import chromadb
from sentence_transformers import SentenceTransformer
from google import genai
from dotenv import load_dotenv
import os
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Open the database created in notebook 01.
db_client = chromadb.PersistentClient(path="./week7_rag_db")
collection = db_client.get_or_create_collection(name="documents")

print("RAG system is ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAG system is ready!


## Step 1: Retrieve context

This is the Week 6 part.

The important difference is that we return the chunks so another function can give them to the LLM.


In [3]:
def retrieve_context(question, top_k=3):
    # Converts the question into a vector.
    question_embedding = embedding_model.encode(question).tolist()

    # Finds the most similar chunks.
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    # Takes only the document text.
    chunks = results["documents"][0]

    # Joins the chunks into one text block called "context".
    context = "\n".join(
        f"Context {i}: {chunk}"
        for i, chunk in enumerate(chunks, start=1)
    )

    return context


## Step 2: Generate an answer

This is the main new application in Week 7.

We send Gemini:
1. The retrieved context.
2. The question.

We also tell gemini to use only the context. This is an attempt to keep the answer **grounded**.


In [ ]:
def generate_answer(question, context):
    system_message = (
        "Answer questions using ONLY the context provided by the user. "
        "Do not use outside knowledge. "
        "If the answer is not in the context, say: "
        "'I don't know based on the given documents.'"
    )

    # f before the string lets Python insert variables such as
    # {context} and {question} into the multi-line text.
    user_message = f"""
Context:
{context}

Question:
{question}

Answer using only the context above.
"""

    # Send the request to Gemini.
    response = client.messages.create(
        model="gemini-3.7-flash",
        max_tokens=300,
        system=system_message,
        messages=[
            {"role": "user", "content": user_message}
        ]
    )

    return response.content[0].text


## Step 3: Put retrieval + generation together

This function is the complete RAG pipeline.


In [5]:
def rag_answer(question, top_k=3):
    # STEP 1: Retrieve useful chunks.
    context = retrieve_context(question, top_k)

    print("Question:")
    print(question)

    print("\nRetrieved context:")
    print(context)

    # STEP 2: Generate an answer from that context.
    answer = generate_answer(question, context)

    print("\nFinal answer:")
    print(answer)

    return answer


In [6]:
rag_answer("How can AI help create a timetable?")

Question:
How can AI help create a timetable?

Retrieved context:
Context 1: AI scheduling can assign teachers, rooms, courses, and time slots while following constraints.
Context 2: Artificial Intelligence allows computers to perform tasks that normally require human intelligence.
Context 3: Machine learning allows a computer to learn patterns from examples and data.


AttributeError: 'Client' object has no attribute 'messages'

In [7]:
rag_answer("What is semantic search?")
rag_answer("How does a genetic algorithm improve a solution?")


Question:
What is semantic search?

Retrieved context:
Context 1: Semantic search retrieves information based on meaning rather than exact keywords.
Context 2: Vector databases store embeddings and support semantic similarity search.
Context 3: Natural language processing helps computers understand and generate human language.


AttributeError: 'Client' object has no attribute 'messages'

## What I built

**Question**
-> **retrieve_context()**
-> **Retrieved context**
-> **generate_answer()**
-> **Final answer**

That is the basic RAG pipeline.
